# Setup Modal app from Modal notebooks!

Assuming there is a persistent volume "output" which already mounted to this Modal notbooks.

In [ ]:
%%bash
if [ ! -d /mnt/output/manim-trainer ]; then
  git clone https://github.com/khieunguyen/manim-trainer.git /mnt/output/manim-trainer
else
  cd /data/manim-trainer
  git pull
fi


chmod +X /mnt/output/manim-trainer/scripts/setup_env.sh
chmod +X /mnt/output//manim-trainer/scripts/train_manim.sh
chmod +X /mnt/output//manim-trainer/scripts/train_manim_skip_sft.sh

cp /mnt/output/manim-trainer/scripts/setup_env.sh .
chmod +X setup_env.sh

### FineTune Qwen 3.6 model

In [ ]:
%%bash
modal run --detach /mnt/output/manim-trainer/scripts/train_modal.py --task train_manim

### FineTune GRPO only (skip SFT step - using previous SFT finetuned model)

Make sure you updated the correct path of the finetuned SFT model in "/mnt/output/manim-trainer/scripts/train_manim_skip_sft.sh"

In [ ]:
%%bash
modal run --detach /mnt/output/manim-trainer/scripts/train_modal.py --task train_skip_sft

In [ ]:
%%bash
modal run --detach /mnt/output/manim-trainer/scripts/train_modal.py --task train_skip_sft


In [ ]:
!rm -rf /mnt/output/qwen3.6_27b_manim_sft_v2/Qwen3.6_27B_lora_r64_sft_20260609_171147
!rm -rf /mnt/output/qwen3.6_27b_manim_sft_v2/Qwen3.6_27B_lora_r64_sft_20260609_212756


# Test pipeline

In [ ]:
%%bash
set -e

export LD_LIBRARY_PATH="/usr/local/lib/python3.12/site-packages/nvidia/cu13/lib:$LD_LIBRARY_PATH"

cd /root/manim-trainer

python manim_trainer.py grpo-trainer train \
  --train-model "unsloth/gemma-4-12b-it" \
  --load-in-4bit \
  --sft-epochs 1 \
  --grpo-epochs 0 \
  --max-seq-length 2048 \
  --prompt-portion 0.2 \
  --lora-rank 16 \
  --per-device-train-batch-size 16 \
  --gradient-accumulation-steps 1 \
  --train-data-path "data/manim_sft_dataset_train_v2.parquet" \
  --test-data-path "data/manim_sft_dataset_test_v2.parquet" \
  --learning-rate 2e-6 \
  --model-loader-type "auto" \
  --random-state 1230 \
  --output-dir "output/gemma4_12b_manim_sft_test_40gb" \
  --model-list-file "output/gemma4_12b_manim_sft_test_40gb/trained_model_list.txt"

In [4]:
!echo 'export LD_LIBRARY_PATH=/usr/local/lib/python3.12/site-packages/nvidia/cu13/lib:$LD_LIBRARY_PATH' >> ~/.bashrc

# FineTune Gemma 4

In [ ]:
%%bash
set -e

export LD_LIBRARY_PATH="/usr/local/lib/python3.12/site-packages/nvidia/cu13/lib:$LD_LIBRARY_PATH"

cd /root/manim-trainer

python manim_trainer.py grpo-trainer train \
  --train-model "unsloth/gemma-4-31B-it" \
  --load-in-4bit \
  --sft-epochs 3 \
  --grpo-epochs 1 \
  --max-seq-length 4096 \
  --prompt-portion 0.2 \
  --lora-rank 64 \
  --per-device-train-batch-size 16 \
  --gradient-accumulation-steps 1 \
  --train-data-path "data/manim_sft_dataset_train_v2.parquet" \
  --test-data-path "data/manim_sft_dataset_test_v2.parquet" \
  --learning-rate 2e-6 \
  --grpo-learning-rate 5e-7 \
  --grpo-num-generations 8 \
  --grpo-num-iterations 4 \
  --grpo-mode "grpo" \
  --grpo-start-temperature 0.9 \
  --suppress-thinking-in-grpo \
  --no-think-tag "/no_think" \
  --text-reward-n-workers 1 \
  --video-reward-n-workers 8 \
  --model-loader-type "auto" \
  --random-state 1230 \
  --output-dir "/mnt/output/gemma4_31b_it_manim_sft_b16_ga1" \
  --model-list-file "/mnt/output/gemma4_31b_it_manim_sft_b16_ga1/trained_model_list.txt"

# FineTune Qwen 3.6

In [5]:
%%bash
set -e

export LD_LIBRARY_PATH="/usr/local/lib/python3.12/site-packages/nvidia/cu13/lib:$LD_LIBRARY_PATH"

cd /root/manim-trainer

python manim_trainer.py grpo-trainer train \
  --train-model "Qwen/Qwen3.6-27B" \
  --load-in-4bit \
  --sft-epochs 3 \
  --grpo-epochs 1 \
  --max-seq-length 4096 \
  --prompt-portion 0.2 \
  --lora-rank 64 \
  --per-device-train-batch-size 16 \
  --gradient-accumulation-steps 1 \
  --train-data-path "data/manim_sft_dataset_train_v2.parquet" \
  --test-data-path "data/manim_sft_dataset_test_v2.parquet" \
  --learning-rate 2e-6 \
  --grpo-learning-rate 5e-7 \
  --grpo-num-generations 8 \
  --grpo-num-iterations 4 \
  --grpo-mode "grpo" \
  --grpo-start-temperature 0.9 \
  --suppress-thinking-in-grpo \
  --no-think-tag "/no_think" \
  --text-reward-n-workers 1 \
  --video-reward-n-workers 8 \
  --model-loader-type "auto" \
  --random-state 1230 \
  --output-dir "/mnt/output/qwen3.6_27b_manim_sft_v2" \
  --model-list-file "/mnt/output/qwen3.6_27b_manim_sft_v2/trained_model_list.txt" \
 

echo "Training started with PID $!"

Training started with PID 5303
